# Reading a drought from space — walkthrough

The whole analysis, step by step: which pixels to trust, how grassland
condition is measured, what the weather stations say, and how mowing is
detected and validated.

**This notebook runs on synthetic data by default**, so it runs on a fresh
clone with no account and no download. The synthetic inputs are calibrated to
the real data (same schemas, parcel shapes and headline numbers). This copy
shows the numbers and tables of a synthetic run; the figures appear when you
run it, stamped SYNTHETIC. The **real results** are quoted in the text beside
each step, and the real figures are in [docs/gallery.md](../docs/gallery.md).

All logic lives in `src/drought/`; this notebook only calls it. All settings
are in `config.yaml`. The reasoning behind each choice is in
[docs/METHOD.md](../docs/METHOD.md).

| | |
|---|---|
| Synthetic (default) | `make install`, then run all cells. About 2 minutes. |
| Real data | Set `REAL = True` below. Needs the inputs in `data/` ([WORKFLOW](../docs/WORKFLOW.md)). Never commit a notebook run on real data; a test refuses it. |

In [1]:
REAL = False          # True: your own data in data/ (see docs/WORKFLOW.md)
SITE = "linthebene"   # the only site with hand labels

import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib
matplotlib.use("Agg")                 # figures are saved, then shown below
import io
from IPython.display import Image, display
from PIL import Image as PILImage
import pandas as pd

from drought import config
config.activate("real" if REAL else "synthetic")   # before any other drought import
if not REAL:
    from drought import synthetic
    synthetic.stamp_figures()
    if not synthetic.exists():
        synthetic.generate()

def show(path, px=1100):
    # show a saved figure, downscaled so the notebook stays small
    im = PILImage.open(path)
    im.thumbnail((px, px))
    buf = io.BytesIO()
    im.save(buf, format="PNG", optimize=True)
    display(Image(data=buf.getvalue()))

print("profile:", config.active_profile())

profile: synthetic


## 1. Which pixels can be trusted

Swiss grassland fields are small: the median field is a few tenths of a
hectare, and most of its 10 m Sentinel-2 cells touch an edge. A cell on a field
edge mixes meadow with road, hedge or arable land.

So the mask is built from a physical model, not a rule of thumb. The sensor's
response is a detector box convolved with Gaussian optics, fitted to the
system MTF at Nyquist, which ESA specifies as 0.15–0.30 for the 10 m bands;
0.25 is used here. A cell's **purity** is the share of its response that comes
from inside a field of the right type. Only cells with purity ≥ 0.90 are used.

In [2]:
from drought import mask, mask_figures
mask_figures.figure_psf()                # the sensor response, and what it implies
show(config.figures_dir("mask") / "psf.png")

  sigma 4.35 m, FWHM 12.7 m  ->  output/synthetic/figures/mask/psf.png


### 1.1 Which fields: the master parcel set

Fields are only compared across years if they were grassland of the same
management type (intensive or extensive) **in every year**. Parcel outlines are
redrawn, split and merged between yearly deliveries, so they are matched by
area overlap, not by ID. Matching by centroid had kept 43 % of fields; overlap
keeps 81 %.

*Real data: 66,843 parcels → 54,332 kept (81.3 %, 83,662 ha).*

In [3]:
master = mask.build_mask()
areas = {n: float(master[master["_code"].isin(c)].geometry.area.sum() / 1e4)
         for n, c in mask.GROUPS.items()}

  8,315 grassland parcels in 2025 (slivers below 0.05 ha dropped)
  overlaying 2017...

 1 s
  overlaying 2018...

 1 s
  overlaying 2019...

 1 s
  overlaying 2020...

 1 s
  overlaying 2021...

...

  mown            5,500       5,877      8,707   1.48        4,278   0.73


  grazed          1,111       4,621      6,054   1.31        4,055   0.88


  intensive       4,033       5,377      8,268   1.54        3,987   0.74


  extensive       1,462         444      1,413   3.18          161   0.36

  polygon ha: inside the raster window. The ratio at >0 MUST exceed 1:
  the PSF response spreads beyond every boundary. If it does not,
  something is wrong.



  wrote data/synthetic/masks/grassland_mask_sg.tif (6.2 MB)
  band order: b1 mown, b2 grazed, b3 intensive, b4 extensive

  FLOAT purity 0-1. No persistence multiplication — persistence is already in
  the parcel selection. Threshold in GEE with .gte(0.9), and upload with
  pyramidingPolicy {'.default': 'sample'} or reduce at the native 10 m.


### 1.2 What the threshold costs

The area check is also an invariant. At purity > 0 the observed area must
*exceed* the polygon area, because the response reaches past every boundary.
When it once fell below 1, that exposed a mask that had zeroed the halo
(METHOD §6). At ≥ 0.90, intensive meadows keep about 80 % of their area and
extensive ones under half. Extensive fields are smaller and more elongated.

In [4]:
area = mask_figures.area_check(master=master)
mask_figures.figure_parcel_size(master=master)
show(config.figures_dir("mask") / "area_check_all.png")
show(config.figures_dir("mask") / "parcel_size.png")

full extent  ·  master parcel set  ·  reference clipped to the raster extent

  category      parcels  polygon ha  median ha   shape
  mown            5,500       5,877       0.45    5.76
  grazed          1,111       4,621       0.60    6.31
  intensive       4,033       5,377       0.70    5.54
  extensive       1,462         444       0.20    6.35

  observed area [ha] and ratio, by purity threshold:
  category                  >0           >0.25            >0.5           >0.75            >0.9
  mown            8,707   1.48    6,759   1.15    6,061   1.03    5,113   0.87    4,278   0.73
  grazed          6,054   1.31    4,941   1.07    4,653   1.01    4,322   0.94    4,055   0.88
  intensive       8,268   1.54    6,139   1.14    5,500   1.02    4,692   0.87    3,987   0.74
  extensive       1,413   3.18      600   1.35      430   0.97      267   0.60      161   0.36

  the ratio at >0 must exceed 1 — the PSF spreads past every boundary.



  -> output/synthetic/figures/mask/area_check_all.png
  group         parcels  median ha  <0.5 ha  median shape  pixels in a median field
  intensive       4,033       0.70      42%          5.54                        70
  extensive       1,462       0.20      84%          6.35                        20
  mown            5,500       0.45      53%          5.76                        45
  grazed          1,111       0.60      45%          6.31                        60

  196 of 12,106 parcels (1.6%) have a shape index above 16 and are clipped in the right panel only.



  -> output/synthetic/figures/mask/parcel_size.png


In [5]:
mask_figures.figure_purity_explainer(site=SITE, group="intensive", areas=areas)
show(config.figures_dir("mask") / f"purity_explainer_{SITE}_intensive.png")
if not REAL:   # no real parcel map is published, by design
    mask_figures.figure_purity_map(SITE, group="intensive")
    show(config.figures_dir("mask") / f"purity_map_{SITE}_intensive.png")

  -> output/synthetic/figures/mask/purity_explainer_linthebene_intensive.png


  -> output/synthetic/figures/mask/purity_map_linthebene_intensive.png


### 1.3 Is the threshold a lever on the answer?

The threshold was swept from 0 to 0.95 in Earth Engine (`drought ee sensitivity`).
*Real data:* the 2026 anomaly is negative at every threshold, at every site.
It grows steadily as the threshold tightens: by 3–4 % for intensive meadows
and 9–15 % for extensive ones, the signature of mixed edge pixels diluting the
signal. At 0.90 it is within 0.7–3.3 % of the strictest value, so the reported
drought signal is slightly understated, never inflated (METHOD §2.5).

In [6]:
if REAL:
    from drought import figures
    figures.figure_purity_sensitivity(SITE, regime="extensive")
else:
    print("The sensitivity sweep needs an Earth Engine export; real numbers are quoted above.")

The sensitivity sweep needs an Earth Engine export; real numbers are quoted above.


## 2. Grassland condition

For every clear Sentinel-2 acquisition (Cloud Score+ ≥ 0.60), Earth Engine
averages NDMI over the pure cells of each site and management type. NDMI
responds to canopy water, and saturates far less than NDVI over dense meadow.
2026 is compared with the 2019–2025 envelope. The **season integral** is the
area under the curve from May to September (DOY 121–273), taken only on days
every year observed, so that a sunnier year is not favoured.

*Real data: −27 % / −29 % at Linthebene and −23 % / −28 % in the Rheintal
(intensive / extensive), 2026 last of eight years everywhere.*

In [7]:
from drought import trajectory
tpath = None if REAL else config.ROOT / config.CFG.paths.gee / "grassland_trajectory_multi.csv"
# Alpstein (summering pastures) is extracted too, but not reported: too few shared days.
traj = pd.concat([trajectory.plot(site=s, path=tpath, index="NDMI", verbose=False)
                  for s in ("linthebene", "rheintal")])
for s in ("linthebene", "rheintal"):
    show(config.figures_dir(s) / "vi_trajectory_NDMI_intensive.png")
traj[["site", "regime", "integral_pct", "rank", "n_days"]].round(1)


linthebene / extensive: 256 obs
    window DOY 121-273, 7 baseline years, 2026 runs to DOY 248
    integral over 86 days shared by all 8 years


  -> output/synthetic/figures/linthebene/vi_trajectory_NDMI_extensive.png

linthebene / intensive: 256 obs
    window DOY 121-273, 7 baseline years, 2026 runs to DOY 248
    integral over 86 days shared by all 8 years


  -> output/synthetic/figures/linthebene/vi_trajectory_NDMI_intensive.png



rheintal / extensive: 227 obs
    window DOY 121-273, 7 baseline years, 2026 runs to DOY 246
    integral over 94 days shared by all 8 years


  -> output/synthetic/figures/rheintal/vi_trajectory_NDMI_extensive.png

rheintal / intensive: 236 obs
    window DOY 121-273, 7 baseline years, 2026 runs to DOY 246
    integral over 94 days shared by all 8 years


  -> output/synthetic/figures/rheintal/vi_trajectory_NDMI_intensive.png


,site,regime,integral_pct,rank,n_days
0,linthebene,extensive,-28.9,1,86
1,linthebene,intensive,-27.3,1,86
0,rheintal,extensive,-28.2,1,94
1,rheintal,intensive,-22.7,1,94


## 3. An independent check: the weather station

MeteoSwiss station St. Gallen, the same series for the current year and the
1991–2020 reference. The second panel is the climatic water balance P − ET0,
with ET0 estimated by Hargreaves from daily temperatures.

*Real data: 690 mm by 12 October, 61 % of the median and the driest of 31
years; P − ET0 −74 mm against a median of +494 mm.*

In [8]:
from drought import precip
if REAL:
    p = precip.main()
else:
    daily, abbr, lat = precip.load_local(config.station_dir())
    p = precip.main(daily=daily, abbr=abbr, lat=lat, station_label="Synthetic station")
show(p["file"])
{k: v for k, v in p.items() if k != "file"}

  13,069 daily records, 1991-01-01 -> 2026-10-12
  current year complete through aligned DOY 285
  baseline precipitation: no missing days
  2026 precipitation: no missing days

Precipitation to DOY 285: 689 mm (61% of the 1991-2020 median), rank 1 of 31 IN THE COMPARISON WINDOW (not the full station record)
Water balance to DOY 285: -75 mm (median 494 mm), rank 1


  -> output/synthetic/figures/station/precip_syn_2026.png


{'end_doy': 285,
 'precip_mm': 689.4,
 'precip_median_mm': 1134.9999999999998,
 'precip_pct_of_median': 60.74008810572688,
 'precip_rank': 1,
 'n_years': 31,
 'wb_mm': -74.5007383239109,
 'wb_median_mm': 493.8966382788891,
 'wb_rank': 1}

## 4. Did farmers stop mowing earlier?

A cut shows as an abrupt drop in a field's NDMI. The detector is a transparent
rule set, tuned on 42 hand-labelled field-years and then **frozen**. Its rules,
and why each exists, are in `src/drought/detect.py` and METHOD §5.2.

In [9]:
import drought as dr
from drought import detect, evaluate, labels, metrics, figures

df = dr.load_site(SITE, verbose=False, labelled=REAL)
ev = detect.detect_all(df, verbose=False)
lab = labels.check_labels(labels.load_labels(SITE), df)
print(f"{df.parcel_id.nunique()} fields, {len(lab)} labelled field-years")

loaded 143 parcel-years, 452 cuts from data/synthetic/labels/cut_labels_linthebene.json


  143 parcel-years usable, 452 cuts
  parcel-years per year: 2019:18, 2020:17, 2021:17, 2022:19, 2023:17, 2024:17, 2025:21, 2026:17
80 fields, 143 labelled field-years


### 4.1 Look before counting

One field, every year: the red lines are the detector's cuts. The question for
each panel is whether there is a line at every drop your eye sees, and nowhere
else.

In [10]:
pid = sorted(k.split("|")[0] for k in lab if k.endswith(f"|{config.CFG.project.current_year}"))[0]
figures.plot_parcel(df, pid, ev, site=SITE, save=True)
show(config.figures_dir(SITE) / f"parcel_{str(pid)[-8:]}.png")

  -> output/synthetic/figures/linthebene/parcel_N-180145.png


### 4.2 Validation, as a supervised-learning problem without a model

Each detection is matched one-to-one to a hand label within 12 days. The
headline compares *years*, so what matters is whether the error rate is the
same in every year.

*Real data (143 labelled field-years, 136 scorable): F1 0.868, and 0.897
without 2026. In 2026 precision falls to 0.64, while recall holds.*

In [11]:
evaluate.score(lab, df)
evaluate.score_by_year(lab, df)[["n_parcel_years", "tp", "fp", "fn", "precision", "recall"]]

  labelled cuts 452 | detected 380
  precision 0.947  recall 0.796  F1 0.865


,n_parcel_years,tp,fp,fn,precision,recall
year,,,,,,
2019,18,51,2,6,0.962,0.895
2020,17,43,7,10,0.860,0.811
2021,17,54,2,15,0.964,0.783
2022,19,55,3,6,0.948,0.902
2023,17,46,1,14,0.979,0.767
2024,17,34,4,13,0.895,0.723
2025,21,56,1,17,0.982,0.767
2026,17,21,0,11,1.000,0.656


### 4.3 Where the errors fall

The synthetic data had predicted a failure before the real labels confirmed
it. A drought trough followed by a steep recovery looks exactly like a cut and
regrowth.

*Real data: 23 of the 34 false positives in 2026 fall between DOY 220 and 260,
the trough and the recovery.* So the last-cut comparison ends at DOY 228 for
every year, and it is computed from the hand labels alone.

On the synthetic data in this notebook, the 2026 season ends in early
September, before the recovery, so the table below shows no such cluster. The
synthetic full-season experiment that predicted it is in METHOD §5.6.

In [12]:
fp = evaluate.false_positive_positions(lab, df)
cur = config.CFG.project.current_year
fp["when"] = pd.cut(fp.doy, range(120, 321, 20))
fp.assign(year=fp.year.eq(cur).map({True: str(cur), False: "other years"})) \
  .pivot_table(index="when", columns="year", values="doy", aggfunc="count", observed=False) \
  .reindex(columns=[str(cur), "other years"]).fillna(0).astype(int)

year,2026,other years
when,,
"(120, 140]",0,1
"(140, 160]",0,3
"(160, 180]",0,2
"(180, 200]",0,3
"(200, 220]",0,2
"(220, 240]",0,1
"(240, 260]",0,4
"(260, 280]",0,0
"(280, 300]",0,0


### 4.4 The result, from the hand labels alone

Each field's last labelled cut up to DOY 228 in 2026, against its own mean in
the earlier years. No detector is involved.

*Real data: the median is 11.9 days earlier, and 15 of 22 fields stopped
sooner. With mostly one baseline year per field, that is an indication rather
than a finding (sign test p ≈ 0.13).*

In [13]:
t = metrics.labels_last_cut_paired(labels.load_labels(SITE), end=228)
print(f"{len(t)} paired fields, median shift {t.shift_days.median():+.1f} days, "
      f"{int((t.shift_days < 0).sum())} stopped sooner")
t.round(1)

loaded 143 parcel-years, 452 cuts from data/synthetic/labels/cut_labels_linthebene.json
15 paired fields, median shift -9.0 days, 12 stopped sooner


,parcel_id,baseline,current,n_baseline_years,shift_days
0,SYN-180145,185.0,176.0,3,-9.0
1,SYN-180170,180.5,176.0,2,-4.5
2,SYN-181804,181.5,186.0,2,4.5
3,SYN-181908,211.0,176.0,1,-35.0
4,SYN-181972,169.0,161.0,1,-8.0
5,SYN-182458,201.3,181.0,3,-20.3
6,SYN-182569,185.0,176.0,2,-9.0
7,SYN-183023,202.0,198.0,2,-4.0
8,SYN-183458,205.5,176.0,2,-29.5
9,SYN-184075,196.5,161.0,2,-35.5


### 4.5 The rejected alternative

A penalised piecewise-linear changepoint fit was built as the principled
alternative and scored on the same labels. It over-segments.

*Real data: F1 0.805 against the heuristic's 0.868.* It stays in the code as a
measured negative result.

In [14]:
from drought import changepoint
changepoint.score_detector(lab, df, changepoint.detect_changepoints)

{'detected': 410,
 'hits': 357,
 'false_pos': 53,
 'missed': 95,
 'precision': 0.871,
 'recall': 0.79,
 'f1': 0.828}

## 5. Every number carries its provenance

Each data file has a record of the settings and mask that produced it, and
`results.json` refuses any section computed under other settings. Results that
outlive the method that made them were this project's most persistent problem
(METHOD §6, bug 6). Now they are refused by construction.

In [15]:
from drought import provenance
src = trajectory._find_csv() if REAL else tpath
rec = provenance.verify(src, "trajectory")      # raises, naming the field, on a mismatch
print("trajectory export matches the current settings and mask")
{k: rec[k] for k in rec if not isinstance(rec[k], (dict, list))}

trajectory export matches the current settings and mask


{'kind': 'trajectory',
 'file': 'data/synthetic/gee/grassland_trajectory_multi.csv',
 'file_sha256': '02092e838858b6cb',
 'settings_hash': '29a017943a4f',
 'profile': 'synthetic',
 'package_version': '0.1.0',
 'created': '2026-10-16T06:38:12+00:00',
 'registered_from': 'earth engine export'}

---
**Next:** [docs/METHOD.md](../docs/METHOD.md) for the reasoning,
[docs/WORKFLOW.md](../docs/WORKFLOW.md) to run on real data.